# Follow-up diagnostics: UNK tokens in the target vocabulary, and where the BLEU/chrF gain comes from

Run order: cells 1-8 rebuild the tokenizer state (CPU is enough, no model needed). Then:

* **Part 1** measures how many target tokens map to `<unk>` in the model's own vocabulary, and which pieces/characters cause it.
* **Part 2** re-scores the saved baseline and extended predictions separately on test sentences whose references contain one of the 95 added words and on those that do not, and adds a paired bootstrap. It needs the two saved prediction CSVs (`baseline_test_predictions.csv` and `extended_model_test_predictions.csv`); attach them as a Kaggle dataset input.

In [ ]:
# Cell 1: install pinned dependencies (same pins as the earlier notebook), then RESTART THE SESSION.
%pip install -q --no-cache-dir "transformers==4.46.3" "tokenizers==0.20.3" "IndicTransToolkit==1.1.1" sentencepiece sacrebleu matplotlib

In [ ]:
# Cell 2: imports, Hugging Face login, output folder
import re, json, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

OUT_DIR = Path("/kaggle/working/vocab_audit")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT_DIR)

In [ ]:
# Cell 3: load the frozen Santali splits (same files as before)
INPUT_ROOT = Path("/kaggle/input")
matches = [p for p in INPUT_ROOT.rglob("santali_train.parquet") if p.parent.name == "splits"]
assert len(matches) == 1, f"Expected one santali_train.parquet, found {len(matches)}"
SPLIT_DIR = matches[0].parent

train_df = pd.read_parquet(SPLIT_DIR / "santali_train.parquet")
val_df   = pd.read_parquet(SPLIT_DIR / "santali_validation.parquet")
test_df  = pd.read_parquet(SPLIT_DIR / "santali_test.parquet")

print("Split sizes (train / val / test):", len(train_df), len(val_df), len(test_df))
print("Expected from earlier work:      17883 / 993 / 994")
print("Columns:", train_df.columns.tolist())

In [ ]:
# Cell 4: load the tokenizer only (no model needed)
from transformers import AutoTokenizer

MODEL_ID = "ai4bharat/indictrans2-indic-indic-1B"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, token=hf_token)

sp = tok.tgt_spm
BASE_TARGET_VOCAB_SIZE = len(tok.tgt_encoder)
EOS_ID, UNK_ID = 2, 3

print("Tokenizer:", type(tok).__name__)
print("Target vocabulary entries:", BASE_TARGET_VOCAB_SIZE, "(earlier notebook: 122672)")
print("Target SentencePiece pieces:", sp.get_piece_size())

## Step 1: rebuild the candidate pool exactly as before
Same procedure as the earlier notebook: whitespace words from the **training** split only, no words made only of punctuation or symbols, more than one SentencePiece piece, frequency of at least 5, then drop forms containing punctuation or symbols. A reproduction check follows; the numbers should match the earlier results.

In [ ]:
# Cell 5: word frequencies and candidate pool (training split only)
train_texts_full = train_df["Santali"].dropna().astype(str).tolist()

word_counts = Counter()
for t in train_texts_full:
    word_counts.update(t.split())
print("Unique whitespace words in training:", len(word_counts), "(earlier: 41757)")

def has_only_punct_symbols(word):
    return all(unicodedata.category(c)[0] in {"P", "S"} for c in word)

def is_word_only(text):
    return bool(text) and all(unicodedata.category(c)[0] not in {"P", "S"} for c in text)

rows = []
for word, freq in word_counts.items():
    if freq < 5 or has_only_punct_symbols(word):
        continue
    pieces = sp.encode(word, out_type=str)
    if len(pieces) > 1:
        rows.append({
            "word": word,
            "frequency": freq,
            "piece_count": len(pieces),
            "existing_pieces": " | ".join(pieces),
            "potential_piece_reduction": len(pieces) - 1,
        })

candidate_df = pd.DataFrame(rows)
clean_candidates = candidate_df[candidate_df["word"].map(is_word_only)].copy()

def select_candidates(min_freq):
    """Return (new_candidates, already_single_token) for a frequency threshold."""
    sub = clean_candidates[(clean_candidates["frequency"] >= min_freq) & (clean_candidates["piece_count"] > 1)]
    sub = (sub.drop_duplicates(subset=["word"])
              .sort_values(["frequency", "potential_piece_reduction", "word"],
                           ascending=[False, False, True], kind="stable")
              .reset_index(drop=True))
    existing = set(tok.tgt_encoder.keys())
    already = sub[sub["word"].isin(existing)].reset_index(drop=True)
    new = sub[~sub["word"].isin(existing)].reset_index(drop=True)
    return new, already

# Reproduction check against the earlier notebook (candidate types per threshold)
expected_types = {5: 3045, 10: 1232, 20: 451, 50: 97, 100: 21}
check = []
for thr, exp in expected_types.items():
    new, already = select_candidates(thr)
    got = len(new) + len(already)
    check.append({"min_freq": thr, "expected_types": exp, "rebuilt_types": got,
                  "match": got == exp, "new_tokens": len(new), "already_single_token": len(already)})
display(pd.DataFrame(check))
if not all(c["match"] for c in check):
    print("WARNING: candidate counts differ from the earlier notebook. Check the dataset version/splits before trusting later results.")

## Step 2: the extension-aware encoder (same logic as the one used in training)
This reproduces the final encoder from the earlier notebook (Section 10.18f plus the punctuation-aware pattern from 9.12): a candidate is replaced by its new ID only if it starts at the beginning of the text or after whitespace and is followed by end-of-text, whitespace or one of `. , ! ? ; : ( ) [ ] { } " '` or the danda marks. The text between matches is encoded with the original tokenizer.

In [ ]:
# Cell 6: encoder factory
def encode_original_target_segment(segment):
    if not segment:
        return []
    with tok.as_target_tokenizer():
        pieces = tok.tokenize(segment)
        return tok.convert_tokens_to_ids(pieces)

def make_extension_encoder(words):
    """Build (encode_fn, word->new_id, compiled pattern) for an ordered list of new words."""
    cand_to_id = {w: BASE_TARGET_VOCAB_SIZE + i for i, w in enumerate(words)}
    pattern = None
    if words:
        ordered = sorted(words, key=lambda w: (-len(w), w))
        pattern = re.compile(
            r"(?<!\S)(" + "|".join(re.escape(w) for w in ordered) +
            r")(?=$|[\s\.,!?;:()\[\]{}\"'।॥])"
        )

    def encode(text, add_eos=False):
        ids, cursor = [], 0
        if pattern is not None:
            for m in pattern.finditer(text):
                ids.extend(encode_original_target_segment(text[cursor:m.start()]))
                ids.append(cand_to_id[m.group(1)])
                cursor = m.end()
        ids.extend(encode_original_target_segment(text[cursor:]))
        if add_eos and (not ids or ids[-1] != EOS_ID):
            ids.append(EOS_ID)
        return ids

    return encode, cand_to_id, pattern

# The 95-token extension used in the training experiment (threshold 50)
new_df_50, already_df_50 = select_candidates(50)
new_words_50 = new_df_50["word"].tolist()
ext_encode, candidate_to_id, ext_pattern = make_extension_encoder(new_words_50)

print("New tokens at threshold 50:", len(new_words_50), "(expected 95)")
print("Already single tokens (not added):", already_df_50["word"].tolist(), "(expected 2)")
print("Extended vocabulary size:", BASE_TARGET_VOCAB_SIZE + len(new_words_50))

# Integrity check: text with no candidate match must encode exactly like the original tokenizer
val_texts = val_df["Santali"].dropna().astype(str).tolist()
checked = mismatches = 0
for t in val_texts:
    if ext_pattern.search(t) is None:
        checked += 1
        if ext_encode(t) != encode_original_target_segment(t):
            mismatches += 1
print(f"Unchanged-text check on validation: {checked} sentences without candidates, {mismatches} mismatches (must be 0)")
assert mismatches == 0

## Part 1: UNK tokens in the target vocabulary
The earlier audit found 893 UNK ids in the 2,000-sentence sample under the model-ID path, in both the original and the extended encoding, while the raw SentencePiece model showed none. Both statements can be true: the SentencePiece model has 128,000 pieces but the model's target vocabulary has only 122,672 entries, so a piece the segmenter picks can be missing from the vocabulary and becomes `<unk>`. This cell finds out which pieces and characters are affected.

In [ ]:
# Cell 9: UNK rate in the model's own target vocabulary, by split, plus the pieces responsible
enc = tok.tgt_encoder
test_texts = test_df["Santali"].dropna().astype(str).tolist()

def target_pieces(text):
    with tok.as_target_tokenizer():
        return tok.tokenize(text)

def unk_stats(texts, name):
    total = unk = sent_with_unk = 0
    unk_pieces = Counter()
    for t in texts:
        ps = target_pieces(t)
        bad = [p for p in ps if p not in enc]       # same rule as convert_tokens_to_ids -> UNK
        total += len(ps); unk += len(bad); sent_with_unk += bool(bad)
        unk_pieces.update(bad)
    print(f"{name:6s} tokens={total:7d}  UNK={unk:5d} ({100*unk/total:.2f}%)  "
          f"sentences with UNK={sent_with_unk}/{len(texts)} ({100*sent_with_unk/len(texts):.1f}%)")
    return unk_pieces

unk_train = unk_stats(train_texts_full, "train")
unk_val   = unk_stats(val_texts, "val")
unk_test  = unk_stats(test_texts, "test")
print("(The winning paper reports 1.69% unknown tokens for Santali with IndicTrans2; compare, but note normalisation may differ.)")

print("\nMost frequent pieces that become UNK in training targets:")
unk_table = pd.DataFrame(unk_train.most_common(30), columns=["piece", "count"])
unk_table["chars_in_piece"] = unk_table["piece"].map(lambda p: " ".join(f"U+{ord(c):04X}" for c in p.replace("\u2581", "")))
display(unk_table)

# Which characters are involved, and does the vocabulary contain them on their own?
char_weight = Counter()
for p, c in unk_train.items():
    for ch in set(p.replace("\u2581", "")):
        char_weight[ch] += c
char_rows = []
for ch, w in char_weight.most_common(25):
    char_rows.append({"char": ch, "codepoint": f"U+{ord(ch):04X}", "name": unicodedata.name(ch, "?"),
                      "weighted_count": w,
                      "char_alone_in_vocab": (ch in enc) or ("\u2581" + ch in enc)})
display(pd.DataFrame(char_rows))

unk_table.to_csv(OUT_DIR / "unk_pieces_train.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(char_rows).to_csv(OUT_DIR / "unk_chars_train.csv", index=False, encoding="utf-8-sig")

## Part 2: where does the BLEU/chrF gain come from?
The 95 added tokens change the target tokenization of only about a third of sentences. Sentences whose reference contains no added word are tokenized identically in both arms. So if the extended model's gain is about the same on the two kinds of sentence, vocabulary extension is unlikely to be the main driver and the differences in training budget and decoding are the first suspects. If the gain is concentrated on sentences with added words, that supports the extension. Either way this is a diagnostic, not proof of cause.

In [ ]:
# Cell 10: subset analysis and paired bootstrap on the SAVED predictions (no model needed)
import glob, sacrebleu

def find_one(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/**/{name}", recursive=True)
    assert hits, f"{name} not found. Attach the saved predictions as a dataset input (or upload them) and rerun."
    return hits[0]

base_df = pd.read_csv(find_one("baseline_test_predictions.csv"), encoding="utf-8-sig")
ext_df  = pd.read_csv(find_one("extended_model_test_predictions.csv"), encoding="utf-8-sig")

refs      = base_df["reference_santali"].fillna("").astype(str).str.strip().tolist()
refs_ext  = ext_df["Reference Santali"].fillna("").astype(str).str.strip().tolist()
base_pred = base_df["prediction_santali"].fillna("").astype(str).str.strip().tolist()
ext_pred  = ext_df["Extended Prediction"].fillna("").astype(str).str.strip().tolist()

assert len(refs) == len(refs_ext) == len(base_pred) == len(ext_pred), "prediction files differ in length"
assert refs == refs_ext, "references differ between the two files; the comparison would not be on the same test set"

def scores(idx):
    r = [refs[i] for i in idx]
    return (sacrebleu.corpus_bleu([base_pred[i] for i in idx], [r]).score,
            sacrebleu.corpus_chrf([base_pred[i] for i in idx], [r]).score,
            sacrebleu.corpus_bleu([ext_pred[i] for i in idx], [r]).score,
            sacrebleu.corpus_chrf([ext_pred[i] for i in idx], [r]).score)

has_added_word = np.array([ext_pattern.search(r) is not None for r in refs])
groups = {"all test sentences": np.arange(len(refs)),
          "reference contains an added word": np.where(has_added_word)[0],
          "reference contains NO added word": np.where(~has_added_word)[0]}

rows = []
for name, idx in groups.items():
    if len(idx) == 0:
        continue
    b_bleu, b_chrf, e_bleu, e_chrf = scores(list(idx))
    rows.append({"subset": name, "n": len(idx),
                 "baseline_BLEU": round(b_bleu, 2), "extended_BLEU": round(e_bleu, 2), "delta_BLEU": round(e_bleu - b_bleu, 2),
                 "baseline_chrF": round(b_chrf, 2), "extended_chrF": round(e_chrf, 2), "delta_chrF": round(e_chrf - b_chrf, 2)})
subset_df = pd.DataFrame(rows)
display(subset_df)
subset_df.to_csv(OUT_DIR / "subset_analysis.csv", index=False)

# How often does the extended model actually produce the added words?
pred_with_added = sum(ext_pattern.search(p) is not None for p in ext_pred)
print(f"Extended predictions containing an added word: {pred_with_added}/{len(ext_pred)}")
print(f"Test references containing an added word:      {int(has_added_word.sum())}/{len(refs)}")

# Paired bootstrap over test sentences (sampling noise only; it cannot test the confounds)
rng = np.random.default_rng(0)
n, B = len(refs), 500
d_bleu, d_chrf = [], []
for _ in range(B):
    idx = rng.integers(0, n, n)
    b_bleu, b_chrf, e_bleu, e_chrf = scores(list(idx))
    d_bleu.append(e_bleu - b_bleu); d_chrf.append(e_chrf - b_chrf)
for name, d in [("BLEU", d_bleu), ("chrF", d_chrf)]:
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"Paired bootstrap, extended minus baseline {name}: mean {np.mean(d):+.2f}, 95% interval [{lo:+.2f}, {hi:+.2f}], "
          f"extended better in {100*np.mean(np.array(d) > 0):.1f}% of resamples")

# Output length comparison on the SAME definition for both systems
def word_len(xs): return sum(len(x.split()) for x in xs)
print("\nHypothesis/reference WORD-length ratio  baseline: %.3f | extended: %.3f" % (word_len(base_pred)/word_len(refs), word_len(ext_pred)/word_len(refs)))
print("Hypothesis/reference CHARACTER-length ratio baseline: %.3f | extended: %.3f" % (sum(map(len, base_pred))/sum(map(len, refs)), sum(map(len, ext_pred))/sum(map(len, refs))))